### Mon but est de repondre à cette question:

## Est ce que les campaignes email augmentent significativement la probabilité d'achat?

## Hypothese 0 : Les emails n'ont aucun influence sur le taux de conversion.
## Hypothese 1 : Les emails augmentent le taux de conversion.

Le groupe de contrôle : No e-mail

Le groupe de traitement : Mens e-mail + Womens e-mail

KPI : Taux de conversion

In [16]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

In [21]:
from statsmodels.stats.proportion import proportions_ztest

In [29]:
from statsmodels.stats.proportion import confint_proportions_2indep


In [3]:
path = kagglehub.dataset_download(
    "bofulee/kevin-hillstrom-minethatdata-e-mailanalytics"
)

print("Dossier :", path)

print("Fichiers :")
for f in os.listdir(path):
    print(f)

Dossier : /Users/madina/.cache/kagglehub/datasets/bofulee/kevin-hillstrom-minethatdata-e-mailanalytics/versions/1
Fichiers :
Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv


In [4]:
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "bofulee/kevin-hillstrom-minethatdata-e-mailanalytics",
    "Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
)

print(df.head())

   recency history_segment  history  mens  womens   zip_code  newbie channel  \
0       10  2) $100 - $200   142.44     1       0  Surburban       0   Phone   
1        6  3) $200 - $350   329.08     1       1      Rural       1     Web   
2        7  2) $100 - $200   180.65     0       1  Surburban       1     Web   
3        9  5) $500 - $750   675.83     1       0      Rural       1     Web   
4        2    1) $0 - $100    45.34     1       0      Urban       0     Web   

         segment  visit  conversion  spend  
0  Womens E-Mail      0           0    0.0  
1      No E-Mail      0           0    0.0  
2  Womens E-Mail      0           0    0.0  
3    Mens E-Mail      0           0    0.0  
4  Womens E-Mail      0           0    0.0  


In [5]:
df['treatment'] = (df['segment'] != 'No E-Mail').astype(int)

### Taux de conversion

In [12]:
conversion_rate = df.groupby('treatment')['conversion'].mean()

print((conversion_rate * 100).round(2))

treatment
0    0.57
1    1.07
Name: conversion, dtype: float64


### Calcul du lift

In [14]:
control = conversion_rate[0]

treatment = conversion_rate[1]

lift = (treatment-control)/control

print((lift * 100).round(2))

86.53


Les clients ayant reçu un email ont un taux de conversion supérieur de 86.5% au groupe contrôle.

### Tableau croisé entre le groupe de traitement et la conversion des clients

In [15]:
pd.crosstab(df['treatment'],df['conversion'])

conversion,0,1
treatment,,
0,21184,122
1,42238,456


### Chi-square test

In [20]:
table = pd.crosstab(df['treatment'], df['conversion'])

chi2, p_value, dof, expected = chi2_contingency(table)


print(p_value)

5.666706148068666e-10


Comme notre p_value < 0.05, on rejette H0, donc les emails augment le taux de conversion.

### intervalle de confiance

In [33]:
control_success = df[df['treatment']==0]['conversion'].sum()
control_total = len(df[df['treatment']==0])

treatment_success = df[df['treatment']==1]['conversion'].sum()
treatment_total = len(df[df['treatment']==1])


result = confint_proportions_2indep(
    treatment_success,
    treatment_total,
    control_success,
    control_total,
    method='wald'
)

result

(np.float64(0.003548426506506275), np.float64(0.006360715804030661))

Avec un intervalle de confiance de 95%, la différence de taux de conversion entre le groupe ayant reçu un email et le groupe contrôle est estimée entre +0,35 et +0,64 point de pourcentage. Comme l'intervalle n'inclut pas zéro, l'effet de la campagne email est statistiquement positif.